In [4]:
from google.colab import files

print("لطفاً این فایل‌ها را انتخاب و آپلود کن:")
print("- car_bama_1_clean_clean.csv")
print("- car_bama_2_clean_clean.csv")
print("- car_divar_1_clean_clean.csv")
print("- car_divar_2_clean_clean.csv")
print("- car_divar_cleaned.csv")

uploaded = files.upload()

print("\nفایل‌های آپلود شده:")
for name in uploaded.keys():
    print(" -", name)


لطفاً این فایل‌ها را انتخاب و آپلود کن:
- car_bama_1_clean_clean.csv
- car_bama_2_clean_clean.csv
- car_divar_1_clean_clean.csv
- car_divar_2_clean_clean.csv
- car_divar_cleaned.csv


Saving car_bama_1_clean_clean.csv to car_bama_1_clean_clean.csv
Saving car_bama_2_clean_clean.csv to car_bama_2_clean_clean.csv
Saving car_divar_1_clean_clean.csv to car_divar_1_clean_clean.csv
Saving car_divar_2_clean_clean.csv to car_divar_2_clean_clean.csv
Saving car_divar_cleaned.csv to car_divar_cleaned.csv

فایل‌های آپلود شده:
 - car_bama_1_clean_clean.csv
 - car_bama_2_clean_clean.csv
 - car_divar_1_clean_clean.csv
 - car_divar_2_clean_clean.csv
 - car_divar_cleaned.csv


In [5]:
# -*- coding: utf-8 -*-
import os
import re
from typing import Dict, List, Callable

import numpy as np
import pandas as pd


# =========================
# 1) توابع کمکی نرمال‌سازی
# =========================

def normalize_digits_fa2en(text: str) -> str:
    """تبدیل ارقام فارسی/عربی به انگلیسی."""
    if text is None:
        return ""
    if isinstance(text, float) and np.isnan(text):
        return ""
    s = str(text)
    trans_table = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
    return s.translate(trans_table)


def normalize_persian_text(text: str) -> str:
    """
    نرمال‌سازی ساده متن فارسی:
    - حذف نیم‌فاصله
    - یکسان‌سازی ی/ک عربی به فارسی
    - کوچک کردن
    - تبدیل چند فاصله به یکی
    """
    if text is None:
        return ""
    if isinstance(text, float) and np.isnan(text):
        return ""

    s = str(text)
    s = s.replace("\u200c", " ").replace("‌", " ")
    s = s.replace("ي", "ی").replace("ك", "ک")
    s = normalize_digits_fa2en(s)
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s


def contains_any(text: str, patterns: List[str]) -> bool:
    """بررسی اینکه آیا یکی از الگوها در متن هست یا نه."""
    return any(p in text for p in patterns)


# =========================
# 2) خواندن و یکسان‌سازی داده‌ها
# =========================

def load_csv_safe(path: str,
                  required: bool = False) -> pd.DataFrame:
    """
    تلاش برای خواندن CSV:
      - اول با utf-8
      - اگر نشد با cp1256
    در صورت خطا:
      - اگر required=True بود، Exception می‌اندازد.
      - اگر required=False بود، یک DataFrame خالی برمی‌گرداند.
    """
    if not os.path.exists(path):
        msg = f"[WARN] File not found: {path}"
        if required:
            raise FileNotFoundError(msg)
        print(msg)
        return pd.DataFrame()

    last_error = None
    for enc in ["utf-8", "cp1256"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"[INFO] Loaded {path} with encoding={enc} and shape {df.shape}")
            return df
        except Exception as e:
            print(f"[WARN] Could not read {path} with encoding={enc}: {e}")
            last_error = e

    msg = f"[ERROR] Could not read {path} with tried encodings."
    if required:
        raise RuntimeError(msg) from last_error
    print(msg)
    return pd.DataFrame()


def get_col(df: pd.DataFrame,
            possible_names: List[str],
            default_value: str = "") -> pd.Series:
    """
    از بین چند اسم ستون احتمالی، اولین ستونی که وجود دارد را برمی‌گرداند.
    اگر هیچ‌کدام نبود، یک ستون پر از default_value برمی‌گرداند.
    """
    for name in possible_names:
        if name in df.columns:
            return df[name]
    return pd.Series([default_value] * len(df), index=df.index)


def unify_bama(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """تبدیل دیتافریم باما به اسکیما مشترک."""
    if df.empty:
        return df

    out = pd.DataFrame(index=df.index)
    out["source"] = source_name

    out["title"] = get_col(df, ["عنوان", "title"]).astype(str)
    out["body_status"] = get_col(df, ["وضعیت بدنه", "body_status"]).fillna("").astype(str)
    out["engine_status"] = ""
    out["chassis_status"] = ""

    desc_col = get_col(df, ["description", "توضیحات", "توضیحات آگهی"], default_value="")
    out["description"] = desc_col.fillna("").astype(str)

    out["year_raw"] = get_col(df, ["سال تولید", "year"]).astype(str)
    out["price_raw"] = get_col(df, ["قیمت", "price"]).astype(str)

    if "price_num" in df.columns:
        out["price"] = df["price_num"]
    else:
        out["price"] = pd.to_numeric(get_col(df, ["price", "قیمت"], default_value=np.nan),
                                     errors="coerce")

    out["mileage_km"] = pd.to_numeric(
        get_col(df, ["کارکرد", "mileage_km"], default_value=np.nan),
        errors="coerce"
    )

    return out


def unify_divar(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """تبدیل دیتافریم دیوار به اسکیما مشترک."""
    if df.empty:
        return df

    out = pd.DataFrame(index=df.index)
    out["source"] = source_name

    out["title"] = get_col(df, ["title", "عنوان"]).astype(str)
    out["body_status"] = get_col(df, ["body_status", "وضعیت بدنه"], "").fillna("").astype(str)
    out["engine_status"] = get_col(df, ["engine_status", "وضعیت فنی"], "").fillna("").astype(str)
    out["chassis_status"] = get_col(df, ["chassis_status", "وضعیت شاسی"], "").fillna("").astype(str)
    out["description"] = get_col(df, ["description", "توضیحات"], "").fillna("").astype(str)

    out["year_raw"] = get_col(df, ["year", "سال تولید"]).astype(str)
    out["price_raw"] = get_col(df, ["price", "قیمت"]).astype(str)
    out["price"] = pd.to_numeric(
        get_col(df, ["price", "قیمت"], default_value=np.nan),
        errors="coerce"
    )
    out["mileage_km"] = pd.to_numeric(
        get_col(df, ["mileage_km", "کارکرد"], default_value=np.nan),
        errors="coerce"
    )

    return out


def build_master_dataframe(file_paths: Dict[str, Dict[str, str]]) -> pd.DataFrame:
    """ساخت دیتافریم کانکت شده از همه فایل‌ها."""
    frames = []

    for name, path in file_paths.get("bama", {}).items():
        df_raw = load_csv_safe(path, required=False)
        df_uni = unify_bama(df_raw, source_name=name)
        if not df_uni.empty:
            frames.append(df_uni)

    for name, path in file_paths.get("divar", {}).items():
        df_raw = load_csv_safe(path, required=False)
        df_uni = unify_divar(df_raw, source_name=name)
        if not df_uni.empty:
            frames.append(df_uni)

    if not frames:
        print("[ERROR] No data loaded from any file.")
        return pd.DataFrame()

    master = pd.concat(frames, ignore_index=True)
    print(f"[INFO] Master dataframe shape: {master.shape}")
    return master


# =========================
# 3) حذف آگهی‌های قسطی / نمایندگی
# =========================

def has_finance_or_dealer_phrases(text: str) -> bool:
    t = normalize_persian_text(text)
    bad_words = [
        "قسط",
        "پیش قسط",
        "پیش‌قسط",
        "پیش پرداخت",
        "پیش‌پرداخت",
        "اقساط",
        "نمایندگی",
        "نمایشگاه",
        "تماس با آقای",
        "تماس با خانم",
    ]
    return any(bw in t for bw in bad_words)


def filter_finance_and_dealer(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    combined = (df["title"].fillna("") + " " + df["description"].fillna("")).astype(str)
    mask_bad = combined.apply(has_finance_or_dealer_phrases)
    print(f"[INFO] Removing {mask_bad.sum()} finance/dealer-like ads")
    return df.loc[~mask_bad].copy()


# =========================
# 4) ساخت text_all
# =========================

def build_text_all(row: pd.Series) -> str:
    parts = [
        row.get("title", ""),
        row.get("body_status", ""),
        row.get("engine_status", ""),
        row.get("chassis_status", ""),
        row.get("description", "")
    ]
    joined = " ".join([str(p) for p in parts if isinstance(p, str)])
    return normalize_persian_text(joined)


def add_text_all_column(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    df["text_all"] = df.apply(build_text_all, axis=1)
    return df


# =========================
# 5) توابع فیچرهای 0/1
# =========================

def feat_darb_rang(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "یک درب رنگ",
        "1 درب رنگ",
        "۱ درب رنگ",
        "دو درب رنگ",
        "2 درب رنگ",
        "۲ درب رنگ",
        "در جلو رنگ",
        "در عقب رنگ",
        "درب رنگ",
        "در رنگ"
    ]
    return int(contains_any(t, patterns))


def feat_darb_taviz(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "درب تعویض",
        "در تعویض",
        "درها تعویض",
        "درها عوض شده",
        "درب عوض شده"
    ]
    return int(contains_any(t, patterns))


def feat_darb_bedun_rang(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "درب بدون رنگ",
        "در بدون رنگ",
        "درب ها بدون رنگ",
        "درها بدون رنگ",
        "درب بی رنگ",
        "در بی رنگ"
    ]
    return int(contains_any(t, patterns))


def feat_kaput_taviz(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "کاپوت تعویض",
        "تعویض کاپوت",
        "کاپوت عوض شده"
    ]
    return int(contains_any(t, patterns))


def feat_kaput_bedun_rang(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "کاپوت بدون رنگ",
        "کاپوت بی رنگ",
        "بدون رنگ کاپوت"
    ]
    return int(contains_any(t, patterns))


def feat_darb_sandogh_rang(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "درب صندوق رنگ",
        "در صندوق رنگ",
        "صندوق رنگ"
    ]
    if contains_any(t, patterns) and "صندوق بدون رنگ" not in t and "صندوق بی رنگ" not in t:
        return 1
    return 0


def feat_darb_sandogh_bedun_rang(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "درب صندوق بدون رنگ",
        "در صندوق بدون رنگ",
        "صندوق بدون رنگ",
        "صندوق بی رنگ"
    ]
    return int(contains_any(t, patterns))


def feat_shasi_zarbe(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "شاسی ضربه",
        "ضربه به شاسی",
        "شاسی خورده",
        "شاسی کج",
        "شاسی آسیب",
        "شاسی شکسته"
    ]
    return int(contains_any(t, patterns))


def feat_shasi_pelomp(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "شاسی پلمپ",
        "پلمپ شاسی",
        "سالم و پلمپ"
    ]
    return int(contains_any(t, patterns))


def feat_shasi_salem(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "شاسی سالم",
        "شاسی ها سالم",
        "شاسی بی ضربه",
        "شاسی ها بی ضربه",
        "شاسی دست نخورده"
    ]
    return int(contains_any(t, patterns))


def feat_saghf_rang(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "سقف رنگ",
        "رنگ سقف"
    ]
    if contains_any(t, patterns) and "سقف بدون رنگ" not in t and "سقف بی رنگ" not in t:
        return 1
    return 0


def feat_saghf_bedun_rang(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "سقف بدون رنگ",
        "سقف بی رنگ"
    ]
    return int(contains_any(t, patterns))


def feat_sotun_salem(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "ستون سالم",
        "ستون ها سالم",
        "ستون بی ضربه",
        "ستون ها بی ضربه",
        "ستون دست نخورده"
    ]
    return int(contains_any(t, patterns))


def feat_sotun_zarbe(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "ستون ضربه",
        "ضربه به ستون",
        "ستون کج",
        "ستون آسیب",
        "ستون ها ضربه"
    ]
    return int(contains_any(t, patterns))


def feat_golgir_rang(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "گلگیر رنگ",
        "رنگ گلگیر"
    ]
    if contains_any(t, patterns) and "گلگیر بدون رنگ" not in t and "گلگیر بی رنگ" not in t:
        return 1
    return 0


def feat_golgir_bedun_rang(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "گلگیر بدون رنگ",
        "گلگیر بی رنگ"
    ]
    return int(contains_any(t, patterns))


def feat_golgir_taviz(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "گلگیر تعویض",
        "تعویض گلگیر",
        "گلگیر عوض شده"
    ]
    return int(contains_any(t, patterns))


def feat_fanni_salem(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "فنی سالم",
        "از نظر فنی سالم",
        "از لحاظ فنی سالم"
    ]
    return int(contains_any(t, patterns))


def feat_fanni_niyaz_tamir(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "فنی نیاز به تعمیر",
        "نیاز به تعمیر فنی",
        "مشکل فنی",
        "ایراد فنی",
        "خرج فنی"
    ]
    return int(contains_any(t, patterns))


def feat_fanni_tamir_shode(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "فنی تعمیر شده",
        "فنی تازه تعمیر",
        "تعمیر اساسی فنی",
        "تعمیر کلی فنی"
    ]
    if contains_any(t, patterns) or ("فنی" in t and "تعمیر" in t):
        return 1
    return 0


def feat_gearbox_salem(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "گیربکس سالم",
        "گیر بکس سالم",
        "گیربکس بدون مشکل",
        "گیربکس بی صدا"
    ]
    return int(contains_any(t, patterns))


def feat_gearbox_niyaz_tamir(row: pd.Series) -> int:
    t = row["text_all"]
    patterns = [
        "گیربکس نیاز به تعمیر",
        "گیر بکس نیاز به تعمیر",
        "گیربکس خرج",
        "گیر بکس خرج",
        "گیربکس خراب",
        "مشکل گیربکس",
        "ایراد گیربکس"
    ]
    return int(contains_any(t, patterns))


def feat_yek_lakke_rang(row: pd.Series) -> int:
    t = normalize_persian_text(row.get("body_status", "")) + " " + row["text_all"]
    patterns = [
        "یک لکه رنگ",
        "1 لکه رنگ",
        "۱ لکه رنگ"
    ]
    return int(contains_any(t, patterns))


def feat_do_lakke_rang(row: pd.Series) -> int:
    t = normalize_persian_text(row.get("body_status", "")) + " " + row["text_all"]
    patterns = [
        "دو لکه رنگ",
        "2 لکه رنگ",
        "۲ لکه رنگ"
    ]
    return int(contains_any(t, patterns))


def feat_dor_rang(row: pd.Series) -> int:
    t = normalize_persian_text(row.get("body_status", "")) + " " + row["text_all"]
    patterns = [
        "دور رنگ",
        "دوررنگ",
        "تمام رنگ",
        "تمام‌رنگ",
        "تمام رنگی"
    ]
    return int(contains_any(t, patterns))


# =========================
# 6) اعمال فیچرها
# =========================

def get_feature_functions() -> Dict[str, Callable[[pd.Series], int]]:
    return {
        "درب رنگ": feat_darb_rang,
        "درب تعویض": feat_darb_taviz,
        "درب بدون رنگ": feat_darb_bedun_rang,
        "کاپوت تعویض": feat_kaput_taviz,
        "کاپوت بدون رنگ": feat_kaput_bedun_rang,
        "درب صندوق رنگ": feat_darb_sandogh_rang,
        "درب صندوق بدون رنگ": feat_darb_sandogh_bedun_rang,
        "شاسی ضرب خوردگی": feat_shasi_zarbe,
        "شاسی پلمپ": feat_shasi_pelomp,
        "شاسی سالم": feat_shasi_salem,
        "سقف رنگ": feat_saghf_rang,
        "سقف بدون رنگ": feat_saghf_bedun_rang,
        "ستون ها سالم": feat_sotun_salem,
        "ستون ها ضرب دیدگی": feat_sotun_zarbe,
        "گلگیر رنگ شدگی": feat_golgir_rang,
        "گلگیر بدون رنگ": feat_golgir_bedun_rang,
        "گلگیر تعویض": feat_golgir_taviz,
        "فنی سالم": feat_fanni_salem,
        "فنی نیاز به تعمیر": feat_fanni_niyaz_tamir,
        "فنی تعمیر شده": feat_fanni_tamir_shode,
        "گیربکس سالم": feat_gearbox_salem,
        "گیر بکس نیاز به تعمیر": feat_gearbox_niyaz_tamir,
        "یک لکه رنگ": feat_yek_lakke_rang,
        "دو لکه رنگ": feat_do_lakke_rang,
        "دور رنگ": feat_dor_rang,
    }


def apply_feature_functions(df: pd.DataFrame,
                            feature_funcs: Dict[str, Callable[[pd.Series], int]]) -> pd.DataFrame:
    if df.empty:
        print("[WARN] Empty dataframe passed to apply_feature_functions")
        return df

    df = df.copy()
    for col_name, func in feature_funcs.items():
        try:
            print(f"[INFO] Computing feature '{col_name}' ...")
            df[col_name] = df.apply(func, axis=1).astype(int)
        except Exception as e:
            print(f"[ERROR] Failed to compute feature '{col_name}': {e}")
            df[col_name] = 0
    return df


# =========================
# 7) ذخیرهٔ خروجی
# =========================

def save_output(df: pd.DataFrame, path: str) -> None:
    """ذخیره CSV با utf-8-sig تا در Excel فارسی درست دیده شود."""
    try:
        df.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"[INFO] Saved output to {path} with shape {df.shape}")
    except Exception as e:
        print(f"[ERROR] Could not save output to {path}: {e}")


In [6]:
# تعریف مسیر فایل‌ها مطابق نام‌هایی که آپلود کردی
file_paths = {
    "bama": {
        "bama1": "car_bama_1_clean_clean.csv",
        "bama2": "car_bama_2_clean_clean.csv",
    },
    "divar": {
        "divar1": "car_divar_1_clean_clean.csv",
        "divar2": "car_divar_2_clean_clean.csv",
        "divar_clean": "car_divar_cleaned.csv",
    }
}

# 1) ساخت دیتافریم کلی
master = build_master_dataframe(file_paths)
if master.empty:
    print("[FATAL] Master dataframe is empty. بررسی کن فایل‌ها درست آپلود شده باشند.")
else:
    # 2) حذف آگهی‌های قسطی / نمایندگی
    master_filtered = filter_finance_and_dealer(master)

    # 3) اضافه کردن متن تجمیعی
    master_with_text = add_text_all_column(master_filtered)

    # 4) اعمال فیچرها
    feature_funcs = get_feature_functions()
    final_df = apply_feature_functions(master_with_text, feature_funcs)

    # 5) ذخیره خروجی
    output_path = "cars_damage_features_concat_filtered.csv"
    save_output(final_df, output_path)


[INFO] Loaded car_bama_1_clean_clean.csv with encoding=utf-8 and shape (226, 12)
[INFO] Loaded car_bama_2_clean_clean.csv with encoding=utf-8 and shape (620, 13)
[INFO] Loaded car_divar_1_clean_clean.csv with encoding=utf-8 and shape (3551, 13)
[INFO] Loaded car_divar_2_clean_clean.csv with encoding=utf-8 and shape (1587, 13)
[INFO] Loaded car_divar_cleaned.csv with encoding=utf-8 and shape (8935, 13)
[INFO] Master dataframe shape: (14919, 10)
[INFO] Removing 866 finance/dealer-like ads
[INFO] Computing feature 'درب رنگ' ...
[INFO] Computing feature 'درب تعویض' ...
[INFO] Computing feature 'درب بدون رنگ' ...
[INFO] Computing feature 'کاپوت تعویض' ...
[INFO] Computing feature 'کاپوت بدون رنگ' ...
[INFO] Computing feature 'درب صندوق رنگ' ...
[INFO] Computing feature 'درب صندوق بدون رنگ' ...
[INFO] Computing feature 'شاسی ضرب خوردگی' ...
[INFO] Computing feature 'شاسی پلمپ' ...
[INFO] Computing feature 'شاسی سالم' ...
[INFO] Computing feature 'سقف رنگ' ...
[INFO] Computing feature 'سقف بد

In [7]:
from google.colab import files
files.download("cars_damage_features_concat_filtered.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>